In [ ]:
import os
import gc
import scanpy as sc
import scvi
from anndata import AnnData

clustered_directory = os.getenv("CLUSTERED_DATA_DIR", ".")
EGC = ["EGC", "Enteric_Glia"]
FILE = ["SCP1038_clustered.h5ad"]

adata_list: list[AnnData] = []
for file in FILE:
    path = os.path.join(clustered_directory, file)
    adata = sc.read_h5ad(path, backed="r")
    print(adata.n_obs)
    adata = adata[adata.obs["celltype"].isin(EGC)].to_memory()
    adata_list.append(adata)

integrated_adata = sc.concat(adata_list)
del adata_list
gc.collect()

/home/sungjune222/projects/scRNA-seq_pipeline/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


343000


1012

In [2]:
integrated_adata.write_h5ad(os.path.join(clustered_directory, "SCP1038_EGCs.h5ad"))

In [ ]:
import scanpy as sc
import os
clustered_directory = os.getenv("CLUSTERED_DATA_DIR", ".")

integrated_adata = sc.read_h5ad(os.path.join(clustered_directory, "SCP1038_.h5ad"), backed="r")
integrated_adata.obs["celltype"].unique()

['EEC', 'paneth', 'epithelial', '+4cell', 'goblet', ..., 'ICC', 'EGC', 'Tuft', 'cd8_tcell', 'enterochromaffin']
Length: 18
Categories (18, object): ['isc', '+4cell', 'paneth', 'goblet', ..., 'ICC', 'muscle', 'neurons', 'EGC']

In [9]:
filtered_adata = integrated_adata[integrated_adata.obs["leiden"].isin(["2", "15"])].to_memory()
filtered_adata.n_obs

1975

In [ ]:
integrated_adata.var["Piezo1"]

Index(['0610007P14Rik', '0610009B22Rik', '0610009L18Rik', '0610009O20Rik',
       '0610010F05Rik', '0610012D04Rik', '0610012G03Rik', '0610025J13Rik',
       '0610030E20Rik', '0610031O16Rik',
       ...
       'mt-Co2', 'mt-Co3', 'mt-Cytb', 'mt-Nd1', 'mt-Nd2', 'mt-Nd3', 'mt-Nd4',
       'mt-Nd4l', 'mt-Nd5', 'mt-Nd6'],
      dtype='object', length=20670)

In [1]:
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot
from cellrank import pl as cr_pl
from cellrank.estimators import GPCCA
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel
from cellrank.models import GAM

series_name = "GSE180759"
clustered_data_location = find_env_dir("CLUSTERED_DATA_DIR")
de_analysis_location = find_env_dir("DESEQ_DIR")

filtered_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
cellrank_analysis_location = find_env_dir("CELLRANK_ANALYSIS_DIR")
cellrank_analysis_location = os.path.join(
    cellrank_analysis_location, series_name
)
os.makedirs(cellrank_analysis_location, exist_ok=True)

/home/sungjune222/projects/protocols/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Calculating connectivity graph between leiden clusters using PAGA
sc.tl.paga(filtered_adata, groups="leiden")
# Calculating Diffusion Pseudotime (DPT) for ordering cells along developmental trajectories
root_cell_index = np.where(filtered_adata.obs["leiden"] == "5")[0][0]
filtered_adata.uns["iroot"] = root_cell_index
sc.tl.diffmap(filtered_adata)
sc.tl.dpt(filtered_adata)

In [3]:
# CellRank-based Trajectory Inference and Gene Trend Analysis
pseudotime_kernel = PseudotimeKernel(filtered_adata, time_key='dpt_pseudotime')
pseudotime_kernel.compute_transition_matrix()


  0%|          | 0/66432 [00:00<?, ?cell/s]

100%|██████████| 66432/66432 [00:17<00:00, 3766.31cell/s] 


PseudotimeKernel[n=66432, dnorm=False, scheme='hard', frac_to_keep=0.3]

In [4]:
connectivity_kernel = ConnectivityKernel(filtered_adata)
connectivity_kernel.compute_transition_matrix()

ConnectivityKernel[n=66432, dnorm=True, key='connectivities']

In [5]:
kernel = 0.8 * pseudotime_kernel + 0.2 * connectivity_kernel
kernel.compute_transition_matrix()
kernel.plot_projection(basis='umap', color='leiden', title='Model-based Differentiation Flow', 
                       save = os.path.join(cellrank_analysis_location, "differentiation_flow_umap.svg")) 

saving figure to file /home/sungjune222/projects/protocols/data/cellrank/GSE180759/differentiation_flow_umap.svg


/home/sungjune222/projects/protocols/.pixi/envs/default/lib/python3.14/site-packages/scvelo/plotting/utils.py:1124: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pl.show()


In [6]:
#  Generalized Perron Cluster Cluster Analysis (GPCCA)
gpcca = GPCCA(kernel)
gpcca.fit(cluster_key="leiden", n_states=None)
gpcca.predict_terminal_states()
gpcca.predict_initial_states(allow_overlap=True)

gpcca.compute_fate_probabilities() #type: ignore
gpcca.plot_fate_probabilities( #type: ignore
        same_plot=True,
        basis="umap",
        save=os.path.join(cellrank_analysis_location, "fate_probabilities_umap.svg"),
    )

100%|██████████| 9/9 [00:02<00:00,  3.33/s]
/home/sungjune222/projects/protocols/.pixi/envs/default/lib/python3.14/site-packages/scvelo/plotting/scatter.py:656: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  smp = ax.scatter(
/home/sungjune222/projects/protocols/.pixi/envs/default/lib/python3.14/site-packages/scvelo/plotting/scatter.py:656: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  smp = ax.scatter(


saving figure to file /home/sungjune222/projects/protocols/data/cellrank/GSE180759/fate_probabilities_umap.svg


/home/sungjune222/projects/protocols/.pixi/envs/default/lib/python3.14/site-packages/scvelo/plotting/utils.py:1124: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pl.show()


In [11]:
target_gene = "SNAP25"
trend_model = GAM(filtered_adata)
lineages = gpcca.terminal_states.cat.categories #type: ignore


In [ ]:
from  pipeline.utils.cellrank import cellrank_analysis

cellrank_analysis(opc_adata, series_name, start_cluster="4", target_gene="TCP11L2", group_key="leiden")

In [ ]:
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

series_name = "SCP1038_EGC"
enrichment_analysis_location = find_env_dir("ENRICHMENT_ANALYSIS_DIR")

enrichment = pd.read_csv(os.path.join(enrichment_analysis_location, series_name + "_GSEA_KEGG_summary.csv"), index_col=0)


/home/sungjune222/projects/scRNA-seq_pipeline/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: '/home/sungjune222/projects/scRNA-seq_pipeline/data/enrichment_analysis/SCP1038_EGC_enrichment.csv'

Gene Compare Pseudobulk

In [ ]:
def gene_compare_pseudobulk_deseq2(
    adata: AnnData,
    gene: str,
    threshold: int = 0,
    min_cells: int = 30,
):
    if gene not in adata.var_names:
        raise ValueError(f"{gene} not found in adata.var_names")

    samples = adata.obs["sample"].unique().tolist()

    gene_mat = adata[:, gene].X  # (n_cells, 1)
    assert isinstance(gene_mat, csr_matrix)
    gene_counts = gene_mat.toarray().ravel()
    is_pos_all = gene_counts > threshold

    rows, meta, idx = [], [], []

    for sample in samples:
        mask_sample = adata.obs["sample"] == sample
        mask_pos = mask_sample & is_pos_all
        mask_neg = mask_sample & (~is_pos_all)

        n_pos = int(mask_pos.sum())
        n_neg = int(mask_neg.sum())

        if n_pos < min_cells or n_neg < min_cells:
            continue

        adata_pos = adata[mask_pos]
        assert isinstance(adata_pos.X, csr_matrix)
        rows.append(np.asarray(adata_pos.X.sum(axis=0)).ravel())
        meta.append(
            {"sample": sample, "group": "pos", "n_cells": n_pos, "batch": sample}
        )
        idx.append(f"{sample}_pos")

        adata_neg = adata[mask_neg]
        assert isinstance(adata_neg.X, csr_matrix)
        rows.append(np.asarray(adata_neg.X.sum(axis=0)).ravel())
        meta.append(
            {"sample": sample, "group": "neg", "n_cells": n_neg, "batch": sample}
        )
        idx.append(f"{sample}_neg")

    if len(rows) < 4:
        print(f"Not enough paired samples for {gene} pos/neg (need >=2 samples kept).")
        return None

    counts = pd.DataFrame(rows, index=idx, columns=adata.var_names)
    meta = pd.DataFrame(meta, index=idx)
    print(counts)
    print(meta)

    dds = DeseqDataSet(
        counts=counts,
        metadata=meta,
        design="~ batch + group",
        refit_cooks=True,
        n_cpus=CPU_CORE_COUNT,
    )
    sc.pp.filter_genes(dds, min_cells=1)
    dds.deseq2()

    ds = DeseqStats(
        dds,
        contrast=["group", "pos", "neg"],
        n_cpus=CPU_CORE_COUNT,
    )
    ds.summary()

    res = ds.results_df.copy()
    res["marker_gene"] = gene
    res["contrast"] = "pos_vs_neg"
    res["marker_threshold"] = threshold

    out_prefix = f"{series_name}_{gene}_pos_vs_neg"
    out_path = os.path.join(de_analysis_location, out_prefix + "_deseq2.csv")
    res.to_csv(out_path)

    return res

Maybe? Doublet Removal ver 2

In [ ]:
# # %%  Preprocessing functions
# # Detecting and annotating doublets using scVI and SOLO
def doublet_removal(adata: AnnData) -> AnnData:
    # Select genes which are expressed in at least 10 cells
    filtered_genes_candidate = sc.pp.filter_genes(adata, min_cells=10, inplace=False)
    if filtered_genes_candidate is None:
        raise ValueError("No gene left after filtering")
    filtered_genes, _ = filtered_genes_candidate

    # Select genes with high variance
    # VST (Variance Stabilizing Transformation) algorithm (in this case, seruat_v3) is applied due to high variance in genes with high level of expression
    # Variance Stabilization is needed to select genes with whose variance is higher than expected
    adata_gene_view = adata[:, filtered_genes]
    highly_variable_genes_candidate = sc.pp.highly_variable_genes(
        adata_gene_view,
        n_top_genes=2000,
        subset=False,
        flavor="seurat_v3",
        inplace=False,
    )
    if highly_variable_genes_candidate is None:
        raise ValueError("Failed to compute highly variable genes")
    highly_variable_genes = highly_variable_genes_candidate

    vae_adata = adata_gene_view[
        :, highly_variable_genes["highly_variable"].to_numpy()
    ].copy()

    doublet_batches = vae_adata.obs["doublet_batch"].unique().tolist()
    adata_splits = []

    for doublet_batch in doublet_batches:
        batch_adata = adata[adata.obs["doublet_batch"] == doublet_batch].copy()

        if batch_adata.n_obs < 2000:
            raise ValueError(
                f"Not enough cells in doublet batch {doublet_batch} for doublet removal"
            )
        else:
            batch_adata = doublet_removal_in_batch(batch_adata)

        adata_splits.append(batch_adata)

    return sc.concat(adata_splits)


def doublet_removal_in_batch(adata: AnnData) -> AnnData:
    series_name = adata.obs["series"].iloc[0]
    doublet_batch = adata.obs["doublet_batch"].iloc[0]

    # Single-Cell Variational Inference (scVI) separates technical noise (batch effect, library size) from the data
    # By using ZINB (Zero-Inflated Negative Binominal) distribution, zero-inflation (too many zeros, because negative count is impossible) and overdispersion (dispersion greater than mean value, because certain RNAs exhibit minimal basal trnanscription but undergo rapid induction only under specific conditions) can be handled
    # scVI uses VAE (Variational AutoEncoder) structure, encoder maps data x into latent varaible z and decoder restores data x from latent variable z and technical noises

    # Adds fields to the AnnData objet to make it ready for building a Pytorch neural network
    # By default every cell is considered as a single batch
    scvi.model.SCVI.setup_anndata(adata)

    # Constructs a VAE Pytorch neural network, not yet trained (Weights are initialized with random values)
    # Default dimension of scVI latent space is 10
    vae = scvi.model.SCVI(adata, n_latent=5)
    # Trains a VAE Pytorch neural network
    vae.train(
        accelerator="gpu",
        batch_size=DOUBLET_REMOVAL_VAE_BATCH_SIZE,
        datasplitter_kwargs=DataLoader,
        train_size=0.9,
        check_val_every_n_epoch=1,
    )
    plot.plot_validation_loss(
        vae, series_name + "_" + doublet_batch + "_doublet_validation_loss.svg"
    )

    # SOLO (Single-cell Doublet Detection via Variational Inference) identifies whether the cell is a singlet or a doublet
    # A decoder of VAE is not required because technical noise has already been accounted for and filtered out with the latent representation produced by the encoder

    # Constructs a SOLO instance by attaching a classifier head to the pretrained VAE encoder
    solo = scvi.external.SOLO.from_scvi_model(vae)
    # SOLO utilizes Positive-Unlabeled (PU) learning, incorporating synthetic doublets generated by summing the counts of two randomly selected cells.
    # This could reveal both heterotypic doublets (e.g. T cell + B cell) and homotypic doublets (e.g. T cell + T cell)
    # Trains a SOLO instance
    solo.train(
        accelerator="gpu",
        batch_size=DOUBLET_REMOVAL_SOLO_BATCH_SIZE,
        datasplitter_kwargs=DataLoader,
        train_size=0.9,
        check_val_every_n_epoch=1,
    )
    plot.plot_validation_loss(
        solo, series_name + "_" + doublet_batch + "_solo_validation_loss.svg"
    )
    # 'soft' True returns probabilities for singlet and doublet
    solo_probs = solo.predict(soft=True)
    adata.obs["singlet_probability"] = solo_probs["singlet"]

    return adata

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import scanpy as sc
import scvi
from pipeline.utils import plot
from pipeline.utils.env import find_env_dir

series_name = "SCP1038"
clustered_data_location = find_env_dir("CLUSTERED_DATA_DIR")
scvi_model_location = find_env_dir("SCVI_MODEL_DIR")
de_analysis_location = find_env_dir("DESEQ_DIR")

clustered_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
model = scvi.model.SCVI.load(os.path.join(scvi_model_location, series_name), adata=clustered_adata)
de_result = pd.read_csv(os.path.join(de_analysis_location, series_name + "_de.csv"), index_col=0)

In [ ]:
# Detecting and annotating doublets using SCVI and SOLO
def doublet_removal(adata: AnnData) -> AnnData:
    # Select genes which are expressed in at least 10 cells
    filtered_genes_candidate = sc.pp.filter_genes(adata, min_cells=10, inplace=False)
    if filtered_genes_candidate is None:
        raise ValueError("No gene left after filtering")
    filtered_genes, _ = filtered_genes_candidate

    # Select genes with high variance
    # VST (Variance Stabilizing Transformation) algorithm (in this case, seruat_v3) is applied due to high variance in genes with high level of expression
    # Variance Stabilization is needed to select genes with whose variance is higher than expected
    adata_gene_view = adata[:, filtered_genes]
    highly_variable_genes_candidate = sc.pp.highly_variable_genes(
        adata_gene_view,
        n_top_genes=2000,
        subset=False,
        flavor="seurat_v3",
        inplace=False,
    )
    if highly_variable_genes_candidate is None:
        raise ValueError("Failed to compute highly variable genes")
    highly_variable_genes = highly_variable_genes_candidate

    vae_adata = adata_gene_view[
        :, highly_variable_genes["highly_variable"].to_numpy()
    ].copy()

    # Single-Cell Variational Inference (SCVI) separates technical noise (batch effect, library size) from the data
    # By using ZINB (Zero-Inflated Negative Binominal) distribution, zero-inflation (too many zeros, because negative count is impossible) and overdispersion (dispersion greater than mean value, because certain RNAs exhibit minimal basal trnanscription but undergo rapid induction only under specific conditions) can be handled
    # SCVI uses VAE (Variational AutoEncoder) structure, encoder maps data x into latent varaible z and decoder restores data x from latent variable z and technical noises

    # Adds fields to the AnnData objet to make it ready for building a Pytorch neural network
    # By default every cell is considered as a single batch
    scvi.model.SCVI.setup_anndata(vae_adata)

    # Constructs a VAE Pytorch neural network, not yet trained (Weights are initialized with random values)
    # Default dimension of SCVI latent space is 10
    vae = scvi.model.SCVI(vae_adata, n_latent=10)
    # Trains a VAE Pytorch neural network
    vae.train(
        accelerator="gpu",
        batch_size=DOUBLET_REMOVAL_VAE_BATCH_SIZE,
        datasplitter_kwargs=DataLoader,
        train_size=0.9,
        check_val_every_n_epoch=1,
    )
    plot.plot_validation_loss(vae, series_name, file_info="validation_loss")

    # SOLO (Single-cell Doublet Detection via Variational Inference) identifies whether the cell is a singlet or a doublet
    # A decoder of VAE is not required because technical noise has already been accounted for and filtered out with the latent representation produced by the encoder

    # Constructs a SOLO instance by attaching a classifier head to the pretrained VAE encoder
    solo = scvi.external.SOLO.from_scvi_model(vae)
    # SOLO utilizes Positive-Unlabeled (PU) learning, incorporating synthetic doublets generated by summing the counts of two randomly selected cells.
    # This could reveal both heterotypic doublets (e.g. T cell + B cell) and homotypic doublets (e.g. T cell + T cell)
    # Trains a SOLO instance
    solo.train(
        accelerator="gpu",
        batch_size=DOUBLET_REMOVAL_SOLO_BATCH_SIZE,
        datasplitter_kwargs=DataLoader,
        train_size=0.9,
        check_val_every_n_epoch=1,
    )
    plot.plot_validation_loss(solo, series_name, file_info="solo_validation_loss")
    # 'soft' True returns probabilities for singlet and doublet
    solo_probs = solo.predict(soft=True)
    adata.obs["singlet_probability"] = solo_probs["singlet"]

    del vae, solo, vae_adata
    gc.collect()

    return adata